# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Iqra-Jahangir/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Signal checks (before building the rule)

**Signal A — Staleness** (linked to FlyRank's real `stale_visible_page` flag from the session): the idea is that pages not updated in a long time, but still getting real traffic, are more likely to be declining and worth a refresh review.

**Signal B — CTR vs. position** (linked to FlyRank's real `low_ctr_visible_page` flag): the idea is that pages ranking well should get proportionally more clicks — if CTR is low despite a strong position, something (title/meta/snippet) may be underperforming.

In [6]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(df.shape)

# Signal A: staleness bucket table (only visible pages)
visible = df[df["impressions_90d"] >= 500].copy()
visible["staleness_bucket"] = visible["days_since_last_update"].apply(
    lambda d: "stale (>=180d)" if d >= 180 else "fresh (<180d)"
)

signal_a = visible.groupby("staleness_bucket").agg(
    n=("content_id", "count"),
    pct_declining=("trend_direction", lambda x: (x == "down").mean().round(3))
)
print(signal_a)

(30000, 44)
                      n  pct_declining
staleness_bucket                      
fresh (<180d)     16709          0.595
stale (>=180d)       17          0.941


**Signal A verdict: CONFIRMED** — stale, visible pages show a higher share of `trend_direction == "down"` than fresh visible pages (n and % shown above). Staleness is a real, checkable basis for the rule.

In [7]:
signal_b = df.groupby("position_tier").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).round(3)
print(signal_b)

                   n  avg_ctr
position_tier                
deep            1319    0.150
page_1         11814    0.652
page_3_5        7242    0.222
striking        7304    0.323
top_3           2321    1.484


**Signal B verdict: CONFIRMED** — average CTR rises as position tier improves (numbers shown above), confirming CTR-vs-position is a real signal, not noise.

### The rule

Flag pages that are stale (days_since_last_update >= 180) AND still visible (impressions_90d >= 500) — both signals confirmed above.

- **Score:** impressions_90d
- **Reason code:** stale_visible_page
- **Action:** refresh_review

Only current, observed columns are used — no future-window or label-derived fields go into this rule.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import os

baseline = df.copy()
baseline["is_flagged"] = (
    (baseline["days_since_last_update"] >= 180) &
    (baseline["impressions_90d"] >= 500)
)

flagged = baseline[baseline["is_flagged"]].copy()
flagged["reason_code"] = "stale_visible_page"
flagged["action"] = "refresh_review"
flagged["score"] = flagged["impressions_90d"]

queue = flagged.sort_values("score", ascending=False)[
    ["content_id", "score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "avg_position", "ctr"]
]

os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)

print("Flagged pages:", len(queue))
queue.head(20)

Flagged pages: 17


,content_id,score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
16751,content_cf56e2e2e282,61678,stale_visible_page,refresh_review,61678,194,19.7,0.15
16514,content_7368877ea310,59472,stale_visible_page,refresh_review,59472,194,24.8,0.13
7021,content_1bfaa38ff26c,25715,stale_visible_page,refresh_review,25715,194,22.2,0.23
21268,content_0a91db491d14,13299,stale_visible_page,refresh_review,13299,193,10.5,0.49
11489,content_5feee3994adb,7812,stale_visible_page,refresh_review,7812,194,39.0,0.01
12045,content_c2d929d83eaa,7558,stale_visible_page,refresh_review,7558,193,17.9,0.20
698,content_b16bd7307b39,4590,stale_visible_page,refresh_review,4590,194,31.0,0.00
5327,content_fe16a55cd13d,4556,stale_visible_page,refresh_review,4556,194,16.4,0.33
26810,content_ecb6215e79fd,4429,stale_visible_page,refresh_review,4429,194,25.3,0.38
20837,content_928af3e22c80,1697,stale_visible_page,refresh_review,1697,193,15.8,0.12


## 3. Top-20 review

1. **[content_cf56e2e2e282]** — action: refresh_review, reason: stale_visible_page. Wrong if: recently redesigned but the timestamp wasn't updated.
2. **[content_7368877ea310]** — Wrong if: seasonal page naturally quiet right now, not truly declining.
3. **[content_1bfaa38ff26c]** — Wrong if: a sibling page absorbed this page's traffic (consolidation), not real staleness.
4. **[content_0a91db491d14]** — Wrong if: position is already strong and traffic is stable despite the old timestamp.
5. **[content_5feee3994adb]** — Wrong if: content type doesn't need frequent updates (e.g. reference page).
6. **[content_c2d929d83eaa]** — Wrong if: recent minor edits weren't tracked by days_since_last_update.
7. **[content_b16bd7307b39]** — Wrong if: low engagement is due to intent mismatch, not staleness.
8. **[content_fe16a55cd13d]** — Wrong if: impressions are inflated by a single seasonal spike.
9. **[content_ecb6215e79fd]** — Wrong if: page already has a refresh scheduled elsewhere.
10. **[content_928af3e22c80]** — Wrong if: traffic is stable/up despite staleness — flag may be premature.

## 4. Weak picks + leakage check

**Weak picks:** the rows closest to the impressions_90d = 500 cutoff are the ones I trust least — a small data fluctuation could have kept them out of the queue entirely, so their inclusion is more arbitrary than the top rows with much higher visibility.

**Leakage check:** the rule uses only `days_since_last_update` and `impressions_90d`, both of which are observed, present-moment signals available at review time. No future-window columns and no label-derived fields (like `trend_direction` or `trend_pct`) were used as inputs to the score or the flag condition.

In [9]:
rule_inputs = {"days_since_last_update", "impressions_90d"}
banned_inputs = {"trend_direction", "trend_pct"}

print("Rule inputs used:", rule_inputs)
print("Any banned/label-derived columns used as inputs?", bool(rule_inputs & banned_inputs))

Rule inputs used: {'impressions_90d', 'days_since_last_update'}
Any banned/label-derived columns used as inputs? False


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.